# 05 — New Experiments (Proposal Section 2.1)

Three new experiments beyond the core Section 7 reproduction:

| # | Experiment | Script |
|---|---|---|
| 1 | VGG16 / InceptionV3 architecture generalization | `train_multiarc`, `run_exemplars_multiarc`, `evaluate_multiarc` |
| 2 | CLIP ViT-B/32 vision encoder text-neuron analysis | `run_clip_exemplars`, `run_clip_descriptions`, `plot_clip_analysis` |
| 3 | Layer-depth text-neuron distribution (ResNet18) | `plot_layer_analysis` |

Run cells in order. Each experiment is independent — you can skip any section.

In [ ]:
import os, sys, torch
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd().parent / 'milan'))
os.environ.setdefault('MILAN_DATA_DIR',    str(Path.cwd().parent / 'data'))
os.environ.setdefault('MILAN_MODELS_DIR',  str(Path.cwd().parent / 'models'))
os.environ.setdefault('MILAN_RESULTS_DIR', str(Path.cwd().parent / 'results'))

DATA_DIR    = Path(os.environ['MILAN_DATA_DIR'])
MODELS_DIR  = Path(os.environ['MILAN_MODELS_DIR'])
RESULTS_DIR = Path(os.environ['MILAN_RESULTS_DIR'])
VERSION_DIR = DATA_DIR / 'imagenet-spurious-text' / '50pct'
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

---
## Experiment 3 — Layer-depth analysis (easiest, data already exists)

Reads `descriptions_annotated.csv` and plots text-neuron fraction + word diversity per ResNet18 layer.

In [ ]:
from milan_repro.figures.plot_layer_analysis import plot as plot_layer

plot_layer(
    descriptions_csv=RESULTS_DIR / 'descriptions_annotated.csv',
    out_dir=RESULTS_DIR / 'figs',
)
from IPython.display import Image
Image(str(RESULTS_DIR / 'figs' / 'fig_layer_analysis.png'))

---
## Experiment 1 — VGG16 architecture generalization

### Step 1a: Train VGG16

In [ ]:
import yaml
from milan_repro.train.train_multiarc import train as train_multiarc

cfg = yaml.safe_load(open('../configs/resnet18_appendixE.yaml'))
vgg_ckpt = MODELS_DIR / 'vgg16_spurious.pth'

info = train_multiarc('vgg16', cfg, VERSION_DIR, vgg_ckpt, device=DEVICE)
print('best val loss:', info['best_val_loss'])

### Step 1b: Compute VGG16 exemplars

In [ ]:
from milan_repro.milan_glue.run_exemplars_multiarc import run as run_exemplars_multiarc

vgg_dissect_dir = RESULTS_DIR / 'edit' / 'imagenet-spurious-text' / 'vgg16-50pct'
run_exemplars_multiarc('vgg16', VERSION_DIR, vgg_ckpt, vgg_dissect_dir, device=DEVICE)

### Step 1c: Generate VGG16 descriptions

In [ ]:
from milan_repro.milan_glue.run_descriptions import run as run_descriptions
from milan_repro.editing.identify_text_neurons import annotate

vgg_desc_csv = RESULTS_DIR / 'vgg16_descriptions.csv'
run_descriptions(vgg_dissect_dir, vgg_desc_csv, device=DEVICE)

vgg_ann_csv = RESULTS_DIR / 'vgg16_descriptions_annotated.csv'
annotate(vgg_desc_csv, vgg_ann_csv)

import pandas as pd
df = pd.read_csv(vgg_ann_csv)
print(f"VGG16 text neurons: {df['is_text_neuron'].sum()} / {len(df)}")
df[df['is_text_neuron']].head(5)

### Step 1d: VGG16 ablation experiment

In [ ]:
from milan_repro.editing.evaluate_multiarc import run as run_eval_multiarc

vgg_curve_csv = RESULTS_DIR / 'vgg16_ablation_curve.csv'
run_eval_multiarc(
    arch='vgg16',
    version_dir=VERSION_DIR,
    ckpt_path=vgg_ckpt,
    dissect_dir=vgg_dissect_dir,
    descriptions_csv=vgg_ann_csv,
    out_csv=vgg_curve_csv,
    ablation_max=40, ablation_step=2,
    device=DEVICE,
)

### Step 1e: Architecture comparison figure (ResNet18 vs VGG16)

In [ ]:
from milan_repro.figures.plot_arch_comparison import plot as plot_arch

plot_arch(
    csv_paths=[RESULTS_DIR / 'ablation_curve.csv',
               RESULTS_DIR / 'vgg16_ablation_curve.csv'],
    labels=['ResNet18', 'VGG16'],
    out_path=RESULTS_DIR / 'figs' / 'fig_arch_comparison.pdf',
)
from IPython.display import Image
Image(str(RESULTS_DIR / 'figs' / 'fig_arch_comparison.png'))

---
## Experiment 2 — CLIP ViT-B/32 analysis

Requires `clip` package: `pip install git+https://github.com/openai/CLIP.git`

CLIP is **not fine-tuned** — we probe it zero-shot to see which transformer
blocks respond to the painted corner text.

In [ ]:
from milan_repro.milan_glue.run_clip_exemplars import run as run_clip_exemplars

clip_dissect_dir = RESULTS_DIR / 'edit' / 'clip-vitb32'
run_clip_exemplars(VERSION_DIR, clip_dissect_dir, device=DEVICE)

In [ ]:
from milan_repro.milan_glue.run_clip_descriptions import run as run_clip_descriptions

clip_desc_csv = RESULTS_DIR / 'clip_descriptions.csv'
run_clip_descriptions(clip_dissect_dir, clip_desc_csv, device=DEVICE)

In [ ]:
from milan_repro.figures.plot_clip_analysis import plot as plot_clip

plot_clip(
    descriptions_csv=clip_desc_csv,
    out_dir=RESULTS_DIR / 'figs',
)
from IPython.display import Image
Image(str(RESULTS_DIR / 'figs' / 'fig_clip_analysis.png'))

---
## Summary — compare text-neuron fractions across experiments

In [ ]:
import pandas as pd

rows = []
for label, csv_path in [
    ('ResNet18', RESULTS_DIR / 'descriptions_annotated.csv'),
    ('VGG16',    RESULTS_DIR / 'vgg16_descriptions_annotated.csv'),
    ('CLIP ViT-B/32 (zero-shot)', RESULTS_DIR / 'clip_descriptions.csv'),
]:
    if not csv_path.exists():
        continue
    df = pd.read_csv(csv_path)
    if 'is_text_neuron' not in df.columns:
        from milan_repro.editing.identify_text_neurons import text_neuron_mask
        df['is_text_neuron'] = text_neuron_mask(df['description'])
    n_text = int(df['is_text_neuron'].sum())
    rows.append({'Model': label, 'Total units': len(df),
                 'Text neurons': n_text,
                 'Fraction (%)': f"{n_text/len(df)*100:.1f}"})

pd.DataFrame(rows)